# 카카오 도로명 주소 API
  - 건물명, 명소로부터 도로명 주소 구하기

In [1]:
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
filename

Saving roadkey2.txt to roadkey2.txt


'roadkey2.txt'

In [2]:
with open(filename) as f:
  api_key = f.read()

In [3]:
len(api_key)

32

In [4]:
import requests
from urllib.parse import quote

In [5]:
addr = '서울특별시 종로구 종로1길 36(수송동)'
quote(addr)

'%EC%84%9C%EC%9A%B8%ED%8A%B9%EB%B3%84%EC%8B%9C%20%EC%A2%85%EB%A1%9C%EA%B5%AC%20%EC%A2%85%EB%A1%9C1%EA%B8%B8%2036%28%EC%88%98%EC%86%A1%EB%8F%99%29'

In [6]:
search_url = "https://dapi.kakao.com/v2/local/search/address.json"
url = f'{search_url}?query={quote(addr)}'
url

'https://dapi.kakao.com/v2/local/search/address.json?query=%EC%84%9C%EC%9A%B8%ED%8A%B9%EB%B3%84%EC%8B%9C%20%EC%A2%85%EB%A1%9C%EA%B5%AC%20%EC%A2%85%EB%A1%9C1%EA%B8%B8%2036%28%EC%88%98%EC%86%A1%EB%8F%99%29'

In [8]:
result = requests.get(url,
                      headers = {'Authorization': f'KakaoAK {api_key}'}).json()


In [9]:
result

{'documents': [{'address': {'address_name': '서울 종로구 수송동 146-12',
    'b_code': '1111012400',
    'h_code': '1111061500',
    'main_address_no': '146',
    'mountain_yn': 'N',
    'region_1depth_name': '서울',
    'region_2depth_name': '종로구',
    'region_3depth_h_name': '종로1.2.3.4가동',
    'region_3depth_name': '수송동',
    'sub_address_no': '12',
    'x': '126.978988255925',
    'y': '37.5735051436739'},
   'address_name': '서울 종로구 종로1길 36',
   'address_type': 'ROAD_ADDR',
   'road_address': {'address_name': '서울 종로구 종로1길 36',
    'building_name': '종로구청',
    'main_building_no': '36',
    'region_1depth_name': '서울',
    'region_2depth_name': '종로구',
    'region_3depth_name': '수송동',
    'road_name': '종로1길',
    'sub_building_no': '',
    'underground_yn': 'N',
    'x': '126.978988255925',
    'y': '37.5735051436739',
    'zone_no': '03152'},
   'x': '126.978988255925',
   'y': '37.5735051436739'}],
 'meta': {'is_end': True, 'pageable_count': 1, 'total_count': 1}}

In [17]:
result['documents'][0]['address']['x']

'126.978988255925'

In [16]:
result['documents'][0]['address']['y']

'37.5735051436739'

In [18]:
lng = float(result['documents'][0]['address']['x'])
lat = float(result['documents'][0]['address']['y'])

for문 사용, 위도/경도 추가해보기 실습해보기

In [34]:
import pandas as pd
import json

In [35]:
gu = pd.read_csv('구청들.csv')

In [44]:
lat_list, lng_list = [], []

for i in gu['도로명주소']:
  addr = f'{(i)}'
  quote(addr)
  search_url = "https://dapi.kakao.com/v2/local/search/address.json"
  url = f'{search_url}?query={quote(addr)}'
  result = requests.get(url, headers = {'Authorization': f'KakaoAK {api_key}'}).json()
  lng = float(result['documents'][0]['address']['x'])
  lat = float(result['documents'][0]['address']['y'])
  lat_list.append(lat)
  lng_list.append(lng)

In [45]:
# 공공기관.csv 로 저장하기
gu['위도'] = lat_list
gu['경도'] = lng_list
gu.to_csv('공공기관.csv', index = False)

실습

In [ ]:
# 1. 공공기관 지도표시 - 대학 folium 활용
# 2. 본인고향의 관공서 지도 표시
# 3. 부모님 고향이나 아니면 광주

In [49]:
import folium

In [60]:
# 1. 공공기관 지도표시 - 대학 folium 활용
gogeo = pd.read_csv('공공기관.csv')

In [76]:
gogeo

,장소,도로명주소,위도,경도
0,종로구청,서울특별시 종로구 종로1길 36(수송동),37.573505,126.978988
1,노원구청,서울특별시 노원구 노해로 437(상계동),37.654519,127.056297
2,송파구청,서울특별시 송파구 올림픽로 326(신천동),37.514477,127.105860
3,마포구청,서울특별시 마포구 월드컵로 지하190(성산동),37.563426,126.903357
4,양천구청,서울특별시 양천구 목동동로 105(신정동),37.517075,126.866543
5,강서구청,서울특별시 강서구 남부순환로 208(외발산동),37.546867,126.820414


In [81]:
gogeo_map = folium.Map(location = [37.55,126.98], tiles = "Stamen Terrain", zoom_start = 12)

for name, lat, lng in zip(gogeo.장소, gogeo.위도, gogeo.경도):
  folium.Marker([lat, lng], popup = name).add_to(gogeo_map)

In [ ]:
gogeo_map.save('./gogeo.html')
gogeo_map

In [98]:
# 2. 본인고향의 관공서 지도 표시
addr = '인천광역시 남동구 정각로 29'
quote(addr)
search_url = "https://dapi.kakao.com/v2/local/search/address.json"
url = f'{search_url}?query={quote(addr)}'
url

'https://dapi.kakao.com/v2/local/search/address.json?query=%EC%9D%B8%EC%B2%9C%EA%B4%91%EC%97%AD%EC%8B%9C%20%EB%82%A8%EB%8F%99%EA%B5%AC%20%EC%A0%95%EA%B0%81%EB%A1%9C%2029'

In [ ]:
result = requests.get(url, headers = {'Authorization': f'KakaoAK {api_key}'}).json()
result

In [110]:
b = float(result['documents'][0]['address']['x'])
a = float(result['documents'][0]['address']['y'])

In [ ]:
icn_go = folium.Map(location = [37.4562557,126.7052062], tiles = "Stamen Terrain", zoom_start = 12)
folium.Marker([a, b], popup = '인천광역시청').add_to(icn_go)
icn_go

In [112]:
# 3. 부모님 고향이나 아니면 광주
addr = '한경면 금등2길 42'
quote(addr)
search_url = "https://dapi.kakao.com/v2/local/search/address.json"
url = f'{search_url}?query={quote(addr)}'
url

'https://dapi.kakao.com/v2/local/search/address.json?query=%ED%95%9C%EA%B2%BD%EB%A9%B4%20%EA%B8%88%EB%93%B12%EA%B8%B8%2042'

In [ ]:
result = requests.get(url, headers = {'Authorization': f'KakaoAK {api_key}'}).json()
result

In [114]:
a = float(result['documents'][0]['address']['y'])
b = float(result['documents'][0]['address']['x'])

In [ ]:
cju_go = folium.Map(location = [33.4996213,126.5311884], tiles = "Stamen Terrain", zoom_start = 12)
folium.Marker([a, b], popup = '할머니댁').add_to(cju_go)
cju_go